In [4]:
pip install torch transformers accelerate pandas

In [5]:
!pip install -q transformers accelerate bitsandbytes sentencepiece pandas

In [6]:
import json

FILE_PATH = "/content/dataset_10k.jsonl"

with open(FILE_PATH, "r", encoding="utf-8") as f:
    try:
        data = json.load(f)

    except json.JSONDecodeError:
        f.seek(0)
        data = []

        for line in f:
            line = line.strip()

            if line:
                data.append(json.loads(line))

print("Number of examples:", len(data))
print("\nFirst example:")
print(data[0])

Number of examples: 10265

First example:
{'question_id': 1, 'language': 'English', 'category': 'chrono_questions', 'question': 'Arrange the following events in chronological order: Mexican-American War, Second Balkan War, Gaza war (2023–), War in Afghanistan (2001–2021), Battle of Poitiers', 'expected': 'Battle of Poitiers, Mexican-American War, Second Balkan War, War in Afghanistan (2001–2021), Gaza war (2023–)', 'domain': nan, 'unique_id': '070000000101'}


In [7]:
# Check the structure of your dataset

print("Number of records:", len(data))

print("\nFirst record:")
print(data[0])

print("\nKeys in first record:")
print(data[0].keys())

Number of records: 10265

First record:
{'question_id': 1, 'language': 'English', 'category': 'chrono_questions', 'question': 'Arrange the following events in chronological order: Mexican-American War, Second Balkan War, Gaza war (2023–), War in Afghanistan (2001–2021), Battle of Poitiers', 'expected': 'Battle of Poitiers, Mexican-American War, Second Balkan War, War in Afghanistan (2001–2021), Gaza war (2023–)', 'domain': nan, 'unique_id': '070000000101'}

Keys in first record:
dict_keys(['question_id', 'language', 'category', 'question', 'expected', 'domain', 'unique_id'])


In [8]:
from collections import Counter

language_count = Counter(
    item["language"]
    for item in data
)

category_count = Counter(
    item["category"]
    for item in data
)

print("Languages:")
for language, count in language_count.items():
    print(language, ":", count)

print("\nCategories:")
for category, count in category_count.items():
    print(category, ":", count)

Languages:
English : 1892
Gujarati : 2107
Hindi : 2067
Marathi : 2112
Odia : 2087

Categories:
chrono_questions : 980
factual_questions : 1950
indian_questions : 1135
maths_questions : 875
ner_questions : 805
reasoning_questions : 705
semantically_incorrect_questions : 1825
true_false_questions : 985
word_ordering_questions : 1005


In [9]:
chrono_data = [
    item for item in data
    if item["category"] == "chrono_questions"
]

print("Total chronology questions:", len(chrono_data))

# First 100 for Experiment 1
test_data = chrono_data[:100]

print("Questions selected for Experiment 1:", len(test_data))

Total chronology questions: 980
Questions selected for Experiment 1: 100


In [10]:
for item in test_data[:3]:

    print("ID:", item["question_id"])
    print("Language:", item["language"])
    print("Question:", item["question"])
    print("Expected:", item["expected"])
    print("-" * 80)

ID: 1
Language: English
Question: Arrange the following events in chronological order: Mexican-American War, Second Balkan War, Gaza war (2023–), War in Afghanistan (2001–2021), Battle of Poitiers
Expected: Battle of Poitiers, Mexican-American War, Second Balkan War, War in Afghanistan (2001–2021), Gaza war (2023–)
--------------------------------------------------------------------------------
ID: 1
Language: Gujarati
Question: નીચેની ઘટનાઓને કાલક્રમિક ક્રમમાં ગોઠવો: મેક્સિકન-અમેરિકન યુદ્ધ, બીજું બાલ્કન યુદ્ધ, ગાઝા યુદ્ધ (2023–), અફઘાનિસ્તાનમાં યુદ્ધ (2001–2021), પોઇટિયર્સનું યુદ્ધ
Expected: પોઇટિયર્સનું યુદ્ધ, મેક્સિકન-અમેરિકન યુદ્ધ, બીજું બાલ્કન યુદ્ધ, અફઘાનિસ્તાનમાં યુદ્ધ (2001–2021), ગાઝા યુદ્ધ (2023–)
--------------------------------------------------------------------------------
ID: 1
Language: Hindi
Question: निम्नलिखित घटनाओं को कालानुक्रमिक क्रम में व्यवस्थित करें: मैक्सिकन-अमेरिकी युद्ध, दूसरा बाल्कन युद्ध, गाजा युद्ध (2023–), अफगानिस्तान में युद्ध (2001–2021), पोइटियर्स का

In [11]:
!pip install -q transformers accelerate bitsandbytes sentencepiece

In [12]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto"
)

model.eval()

print("Qwen loaded successfully!")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen loaded successfully!


In [13]:
print(torch.cuda.is_available())

True


In [14]:
def generate_answer(question):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a multilingual question answering system. "
                "Answer the question accurately. "
                "For chronology questions, arrange all events "
                "from earliest to latest. "
                "Return only the chronological order."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False
        )

    input_length = inputs["input_ids"].shape[1]

    generated_tokens = outputs[0][input_length:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

In [15]:
item = test_data[0]

print("LANGUAGE:")
print(item["language"])

print("\nQUESTION:")
print(item["question"])

print("\nEXPECTED:")
print(item["expected"])

print("\nMODEL ANSWER:")
print(generate_answer(item["question"]))

LANGUAGE:
English

QUESTION:
Arrange the following events in chronological order: Mexican-American War, Second Balkan War, Gaza war (2023–), War in Afghanistan (2001–2021), Battle of Poitiers

EXPECTED:
Battle of Poitiers, Mexican-American War, Second Balkan War, War in Afghanistan (2001–2021), Gaza war (2023–)

MODEL ANSWER:
Battle of Poitiers, Mexican-American War, War in Afghanistan (2001–2021), Second Balkan War, Gaza war (2023–)


In [16]:
results = []

for i, item in enumerate(test_data):

    print(
        "Processing",
        i + 1,
        "/",
        len(test_data)
    )

    prediction = generate_answer(
        item["question"]
    )

    results.append({
        "question_id": item["question_id"],
        "language": item["language"],
        "category": item["category"],
        "question": item["question"],
        "expected": item["expected"],
        "prediction": prediction,
        "unique_id": item["unique_id"]
    })

Processing 1 / 100
Processing 2 / 100
Processing 3 / 100
Processing 4 / 100
Processing 5 / 100
Processing 6 / 100
Processing 7 / 100
Processing 8 / 100
Processing 9 / 100
Processing 10 / 100
Processing 11 / 100
Processing 12 / 100
Processing 13 / 100
Processing 14 / 100
Processing 15 / 100
Processing 16 / 100
Processing 17 / 100
Processing 18 / 100
Processing 19 / 100
Processing 20 / 100
Processing 21 / 100
Processing 22 / 100
Processing 23 / 100
Processing 24 / 100
Processing 25 / 100
Processing 26 / 100
Processing 27 / 100
Processing 28 / 100
Processing 29 / 100
Processing 30 / 100
Processing 31 / 100
Processing 32 / 100
Processing 33 / 100
Processing 34 / 100
Processing 35 / 100
Processing 36 / 100
Processing 37 / 100
Processing 38 / 100
Processing 39 / 100
Processing 40 / 100
Processing 41 / 100
Processing 42 / 100
Processing 43 / 100
Processing 44 / 100
Processing 45 / 100
Processing 46 / 100
Processing 47 / 100
Processing 48 / 100
Processing 49 / 100
Processing 50 / 100
Processin

In [17]:
import json

output_file = "/content/experiment1_results.json"

with open(
    output_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:", output_file)

Saved: /content/experiment1_results.json


In [18]:
import pandas as pd

results_df = pd.DataFrame(results)

pd.set_option(
    "display.max_colwidth",
    200
)

results_df[
    [
        "language",
        "question",
        "expected",
        "prediction"
    ]
].head(10)

,language,question,expected,prediction
0,English,"Arrange the following events in chronological order: Mexican-American War, Second Balkan War, Gaza war (2023–), War in Afghanistan (2001–2021), Battle of Poitiers","Battle of Poitiers, Mexican-American War, Second Balkan War, War in Afghanistan (2001–2021), Gaza war (2023–)","Battle of Poitiers, Mexican-American War, War in Afghanistan (2001–2021), Second Balkan War, Gaza war (2023–)"
1,Gujarati,"નીચેની ઘટનાઓને કાલક્રમિક ક્રમમાં ગોઠવો: મેક્સિકન-અમેરિકન યુદ્ધ, બીજું બાલ્કન યુદ્ધ, ગાઝા યુદ્ધ (2023–), અફઘાનિસ્તાનમાં યુદ્ધ (2001–2021), પોઇટિયર્સનું યુદ્ધ","પોઇટિયર્સનું યુદ્ધ, મેક્સિકન-અમેરિકન યુદ્ધ, બીજું બાલ્કન યુદ્ધ, અફઘાનિસ્તાનમાં યુદ્ધ (2001–2021), ગાઝા યુદ્ધ (2023–)","મેક્સિકન-અમેરિકન યુદ્ધ, બીજું બાલ્કન યુદ્ધ, અફઘાનિસ્તાનમાં યુદ્ધ (2001–2021), પોઇટિયર્સનું યુદ્ધ, ગ�"
2,Hindi,"निम्नलिखित घटनाओं को कालानुक्रमिक क्रम में व्यवस्थित करें: मैक्सिकन-अमेरिकी युद्ध, दूसरा बाल्कन युद्ध, गाजा युद्ध (2023–), अफगानिस्तान में युद्ध (2001–2021), पोइटियर्स का युद्ध","पोइटियर्स का युद्ध, मैक्सिकन-अमेरिकी युद्ध, दूसरा बाल्कन युद्ध, अफगानिस्तान में युद्ध (2001–2021), गाजा युद्ध (2023–)","गाजा युद्ध (2023–), मैक्सिकन-अमेरिकी युद्ध, दूसरा बाल्कन युद्ध, पोइटियर्स का युद्ध, अफगानिस्तान में युद्ध (2001–2021)"
3,Marathi,"खालील घटनांची कालक्रमानुसार मांडणी करा: मेक्सिकन-अमेरिकन युद्ध, दुसरे बाल्कन युद्ध, गाझा युद्ध (2023–), अफगाणिस्तानमधील युद्ध (2001–2021), पॉइटियर्सची लढाई","पॉइटियर्सची लढाई, मेक्सिकन-अमेरिकन युद्ध, दुसरे बाल्कन युद्ध, अफगाणिस्तानमधील युद्ध (2001–2021), गाझा युद्ध (2023–)","पॉइटियर्सची लढाई, मेक्सिकन-अमेरिकन युद्ध, अफगाणिस्तानमधील युद्ध, दुसरे बाल्कन युद्ध, गाझा युद्ध (2023–)"
4,Odia,"ନିମ୍ନଲିଖିତ ଘଟଣାଗୁଡ଼ିକୁ କାଳାନୁକ୍ରମିକ କ୍ରମରେ ସଜାନ୍ତୁ: ମେକ୍ସିକାନ୍-ଆମେରିକାନ୍ ଯୁଦ୍ଧ, ଦ୍ୱିତୀୟ ବାଲକାନ୍ ଯୁଦ୍ଧ, ଗାଜା ଯୁଦ୍ଧ (2023–), ଆଫଗାନିସ୍ତାନରେ ଯୁଦ୍ଧ (2001–2021), ପୋଇଟିଅର୍ସ ଯୁଦ୍ଧ","ପୋଇଟିଅର୍ସ ଯୁଦ୍ଧ, ମେକ୍ସିକାନ୍-ଆମେରିକାନ୍ ଯୁଦ୍ଧ, ଦ୍ୱିତୀୟ ବାଲକାନ୍ ଯୁଦ୍ଧ, ଆଫଗାନିସ୍ତାନରେ ଯୁଦ୍ଧ (2001–2021), ଗାଜା ଯୁଦ୍ଧ (2023–)","ମେକ୍ସିକାନ୍-ଆମେରିକାନ୍ ଯୁଦ୍ଧ, ଆଫଗାନିସ୍ତାନରେ ଯୁଦ୍ଧ (2001–2021), ଦ୍ୱିତୀୟ ବାଲକାନ୍ ଯ�"
5,English,"Arrange the following events in chronological order: Russo-Ukrainian War, Kosovo War, Battle of Mohács, Suez Crisis, First Nagorno-Karabakh War","Battle of Mohács, Suez Crisis, First Nagorno-Karabakh War, Kosovo War, Russo-Ukrainian War","Battle of Mohács, Suez Crisis, Kosovo War, Russo-Ukrainian War, First Nagorno-Karabakh War"
6,Gujarati,"નીચેની ઘટનાઓને કાલક્રમિક ક્રમમાં ગોઠવો: રુસો-યુક્રેનિયન યુદ્ધ, કોસોવો યુદ્ધ, મોહાક્સનું યુદ્ધ, સુએઝ સંકટ, પ્રથમ નાગોર્નો-કારાબાખ યુદ્ધ","મોહાક્સનું યુદ્ધ, સુએઝ સંકટ, પ્રથમ નાગોર્નો-કારાબાખ યુદ્ધ, કોસોવો યુદ્ધ, રુસો-યુક્રેનિયન યુદ્ધ","મોહાક્સનું યુદ્ધ, સુએઝ સંકટ, રુસો-યુક્રેનિયન યુદ્ધ, પ્રથમ નાગોર્નો-કારાબાખ યુદ્ધ, કોસોવો યુદ્ધ"
7,Hindi,"निम्नलिखित घटनाओं को कालानुक्रमिक क्रम में व्यवस्थित करें: रूस-यूक्रेनी युद्ध, कोसोवो युद्ध, मोहाक का युद्ध, स्वेज संकट, प्रथम नागोर्नो-काराबाख युद्ध","मोहाक का युद्ध, स्वेज संकट, प्रथम नागोर्नो-काराबाख युद्ध, कोसोवो युद्ध, रूस-यूक्रेनी युद्ध","मोहाक का युद्ध, स्वेज संकट, कोसोवो युद्ध, प्रथम नागोर्नो-काराबाख युद्ध, रूस-यूक्रेनी युद्ध"
8,Marathi,"खालील घटनांची कालक्रमानुसार मांडणी करा: रशिया-युक्रेन युद्ध, कोसोवो युद्ध, मोहाचची लढाई, सुएझ संकट, पहिले नागोर्नो-काराबाख युद्ध","मोहाचची लढाई, सुएझ संकट, पहिले नागोर्नो-काराबाख युद्ध, कोसोवो युद्ध, रशिया-युक्रेन युद्ध","कोसोवो युद्ध, मोहाचची लढाई, सुएझ संकट, पहिले नागोर्नो-काराबाख युद्ध, रशिया-युक्रेन युद्ध"
9,Odia,"ନିମ୍ନଲିଖିତ ଘଟଣାଗୁଡ଼ିକୁ କାଳାନୁକ୍ରମିକ କ୍ରମରେ ସଜାନ୍ତୁ: ଋଷ-ୟୁକ୍ରେନୀୟ ଯୁଦ୍ଧ, କୋସୋଭୋ ଯୁଦ୍ଧ, ମୋହାକ୍ସ ଯୁଦ୍ଧ, ସୁଏଜ୍ ସଙ୍କଟ, ପ୍ରଥମ ନାଗୋର୍ଣ୍ଣୋ-କାରାବାଖ୍ ଯୁଦ୍ଧ","ମୋହାକ୍ସ ଯୁଦ୍ଧ, ସୁଏଜ୍ ସଙ୍କଟ, ପ୍ରଥମ ନାଗୋର୍ଣ୍ଣୋ-କାରାବାଖ୍ ଯୁଦ୍ଧ, କୋସୋଭୋ ଯୁଦ୍ଧ, ଋଷ-ୟୁକ୍ରେନୀୟ ଯୁଦ୍ଧ","ମୋହାକ୍ସ ଯୁଦ୍ଧ, କୋସୋଭୋ ଯୁଦ୍ଧ, ଋଷ-ୟୁକ୍ରେନୀୟ ଯୁଦ୍ଧ, ସୁଏଜ୍ ସଙ୍କଟ, ପ୍ରଥମ ନାଗୋ"


In [19]:
import re


def normalize(text):

    text = str(text).lower()

    # Convert arrows to separators
    text = text.replace("→", ",")

    # Remove numbering such as 1. 2. 3.
    text = re.sub(
        r"\d+\.",
        "",
        text
    )

    # Remove extra spaces
    text = " ".join(text.split())

    return text


results_df["expected_normalized"] = (
    results_df["expected"]
    .apply(normalize)
)

results_df["prediction_normalized"] = (
    results_df["prediction"]
    .apply(normalize)
)


results_df["Correct"] = (
    results_df["expected_normalized"]
    ==
    results_df["prediction_normalized"]
)

In [20]:
import pandas as pd

summary = []

for language in sorted(results_df["language"].unique()):

    language_df = results_df[
        results_df["language"] == language
    ]

    total = len(language_df)

    correct = language_df["Correct"].sum()

    accuracy = correct / total

    hallucination = 1 - accuracy

    summary.append({
        "Language": language,
        "Total": total,
        "Correct": correct,
        "Accuracy": round(accuracy * 100, 2),
        "Hallucination Rate": round(
            hallucination * 100,
            2
        )
    })


summary_df = pd.DataFrame(summary)

summary_df

,Language,Total,Correct,Accuracy,Hallucination Rate
0,English,20,5,25.0,75.0
1,Gujarati,20,0,0.0,100.0
2,Hindi,20,2,10.0,90.0
3,Marathi,20,1,5.0,95.0
4,Odia,20,0,0.0,100.0
